In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/CUAD_v1/master_clauses.csv")
df.shape

(510, 83)

In [2]:
df.columns.tolist()

['Filename',
 'Document Name',
 'Document Name-Answer',
 'Parties',
 'Parties-Answer',
 'Agreement Date',
 'Agreement Date-Answer',
 'Effective Date',
 'Effective Date-Answer',
 'Expiration Date',
 'Expiration Date-Answer',
 'Renewal Term',
 'Renewal Term-Answer',
 'Notice Period To Terminate Renewal',
 'Notice Period To Terminate Renewal- Answer',
 'Governing Law',
 'Governing Law-Answer',
 'Most Favored Nation',
 'Most Favored Nation-Answer',
 'Competitive Restriction Exception',
 'Competitive Restriction Exception-Answer',
 'Non-Compete',
 'Non-Compete-Answer',
 'Exclusivity',
 'Exclusivity-Answer',
 'No-Solicit Of Customers',
 'No-Solicit Of Customers-Answer',
 'No-Solicit Of Employees',
 'No-Solicit Of Employees-Answer',
 'Non-Disparagement',
 'Non-Disparagement-Answer',
 'Termination For Convenience',
 'Termination For Convenience-Answer',
 'Rofr/Rofo/Rofn',
 'Rofr/Rofo/Rofn-Answer',
 'Change Of Control',
 'Change Of Control-Answer',
 'Anti-Assignment',
 'Anti-Assignment-Answer',

In [3]:
pd.set_option("display.max_colwidth", 200)

df[["Filename", "Non-Compete", "Non-Compete-Answer"]].head()

,Filename,Non-Compete,Non-Compete-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_525118_EX-10.B(01)_Content License Agreement.pdf,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_8368347_EX-10.9_Content License Agreement.pdf,[],No
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10.1_Content License Agreement.pdf,[],No
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_EX-10.26_Content License Agreement.pdf,[],No


In [4]:
df["Non-Compete-Answer"].value_counts()

Non-Compete-Answer
No     391
Yes    119
Name: count, dtype: int64

In [5]:
df[df["Non-Compete-Answer"] == "Yes"][["Filename", "Non-Compete"]].head(3)

,Filename,Non-Compete
6,EdietsComInc_20001030_10QSB_EX-10.4_2606646_EX-10.4_Co-Branding Agreement.pdf,"['During the Term, (a) Women.com will not buy, sell, display, distribute advertising from (including, but not limited to, banner ads, buttons, badges, text links, hyperlinks or editorial mentions)..."
8,MusclepharmCorp_20170208_10-KA_EX-10.38_9893581_EX-10.38_Co-Branding Agreement.pdf,"['Products. Any failure of Endorser to disclose such conflicting interests, or any breach of this Section, shall be deemed a material breach of the Agreement.', ""Endorser's duty not to compete wit..."
9,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement.pdf,"['Notwithstanding any portion of the foregoing to the contrary, the Non-Competition Period shall terminate prior to the End Date, and for the avoidance of doubt, no party shall be obligated to com..."


In [6]:
answer_cols = [c for c in df.columns if c.endswith("Answer")]

yes_rate = {c: (df[c] == "Yes").mean() for c in answer_cols}

pd.Series(yes_rate).sort_values(ascending=False).round(2)

Anti-Assignment-Answer                        0.73
Cap On Liability-Answer                       0.54
License Grant-Answer                          0.50
Audit Rights-Answer                           0.42
Termination For Convenience-Answer            0.36
Post-Termination Services-Answer              0.36
Exclusivity-Answer                            0.35
Insurance-Answer                              0.33
Revenue/Profit Sharing-Answer                 0.33
Minimum Commitment-Answer                     0.32
Non-Transferable License-Answer               0.27
Ip Ownership Assignment-Answer                0.24
Change Of Control-Answer                      0.24
Non-Compete-Answer                            0.23
Uncapped Liability-Answer                     0.22
Covenant Not To Sue-Answer                    0.20
Rofr/Rofo/Rofn-Answer                         0.17
Volume Restriction-Answer                     0.16
Competitive Restriction Exception-Answer      0.15
Warranty Duration-Answer       

## CHOSEN RISK TYPES

Six clause types that create financial or strategic risk in most commercial contracts:
Uncapped Liability, Termination for Convenience, Exclusivity, Minimum Commitment,
IP Ownership Assignment, Non-Compete.

Criteria: real business risk, at least ~20% of contracts contain it (enough examples
to test), and distinct from each other. Left out: Anti-Assignment (too routine),
Cap on Liability (overlaps with Uncapped), Renewal Term (not a Yes/No label).

In [7]:
from pathlib import Path

TXT_DIR = Path("../data/raw/CUAD_v1/full_contract_txt")

df["txt_path"] = df["Filename"].str.replace(".pdf", ".txt", regex=False).apply(lambda f: TXT_DIR / f)

df["txt_path"].apply(lambda p: p.exists()).value_counts()

txt_path
False    314
True     196
Name: count, dtype: int64

In [8]:
missing = df.loc[~df["txt_path"].apply(lambda p: p.exists()), "Filename"]
print("Missing examples:", missing.head(3).tolist())

actual = sorted(p.name for p in TXT_DIR.iterdir())
print("Files on disk:", len(actual))
print("Actual examples:", actual[:3])

Missing examples: ['HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_12051356_EX-10.18_Development Agreement.PDF', 'SLINGERBAGINC_05_27_2020-EX-10.7-CONSULTING AGREEMENT.PDF', 'KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSULTING AGREEMENT.PDF']
Files on disk: 510
Actual examples: ['2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt', 'ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT.txt', 'ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT.txt']


In [9]:
df["txt_path"] = df["Filename"].str.replace(r"\.pdf$", ".txt", case=False, regex=True).apply(lambda f: TXT_DIR / f)

df["txt_path"].apply(lambda p: p.exists()).value_counts()

txt_path
True     499
False     11
Name: count, dtype: int64

In [10]:
missing = df.loc[~df["txt_path"].apply(lambda p: p.exists()), "Filename"]

for name in missing:
    print(name)

PLAYAHOTELS&RESORTSNV_03_14_2017-EX-10.22-STRATEGIC ALLIANCE AGREEMENT (Hyatt Ziva Cancun).PDF
MACY'S,INC_05_11_2020-EX-99.4-JOINT FILING AGREEMENT.PDF
MOELIS&CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT.PDF
KALLOINC_11_03_2011-EX-10.1-STRATEGIC ALLIANCE AGREEMENT.PDF'
ELECTRAMECCANICA VEHICLES CORP. - Manufacturing Agreement .PDF
Monsanto Company - SECOND A&R EXCLUSIVE AGENCY AND MARKETING AGREEMENT .PDF
PACIRA PHARMACEUTICALS, INC. - A&R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT .PDF
Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement .PDF
Reinsurance Group of America, Incorporated - A&R REMARKETING  AGREEMENT.PDF
SightLife Surgical, Inc. - STRATEGIC SALES & MARKETING AGREEMENT.PDF
NETGEAR,INC_04_21_2003-EX-10.16-AMENDMENT TO THE DISTRIBUTOR AGREEMENT BETWEEN INGRAM MICRO AND NETGEAR-.pdf


In [13]:
import re
import unicodedata

def simplify(name):
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    name = re.sub(r"\.(pdf|txt)'?\s*$", "", name.strip(), flags=re.IGNORECASE)
    return re.sub(r"[^a-z0-9]", "", name.lower())

lookup = {simplify(p.name): p for p in TXT_DIR.iterdir()}
print("Unique simplified names:", len(lookup))

df["txt_path"] = df["Filename"].apply(lambda f: lookup.get(simplify(f)))
df["txt_path"].notna().value_counts()

Unique simplified names: 510


txt_path
True    510
Name: count, dtype: int64

In [14]:
print("Unmatched contract:", df.loc[df["txt_path"].isna(), "Filename"].tolist())

used = set(df["txt_path"].dropna())
print("Unclaimed file:", [p.name for p in TXT_DIR.iterdir() if p not in used])

Unmatched contract: []
Unclaimed file: []


In [15]:
df["words"] = df["txt_path"].apply(lambda p: len(p.read_text(encoding="utf-8", errors="ignore").split()))

df["words"].describe().round(0)

count      510.0
mean      7861.0
std       8367.0
min        109.0
25%       2457.0
50%       5006.0
75%      10196.0
max      47733.0
Name: words, dtype: float64

## OBSERVATIONS

What the contract lengths tell us:

- The middle contract (50%) is about 5,000 words, roughly 10 pages.

- The average (7,861) is much higher than the median (5,006). That gap means a few very long contracts pull the average up. The longest is 47,733 words, almost 10 times the typical one.

- The shortest is just 109 words. That’s probably a short amendment, not a full contract.

**Why this matters**: cost grows with length. One 47,000-word contract costs about as much as ten typical ones. And very short documents might not contain enough to test the six risks properly.

I am going to go with contracts between 1,000 and 15,000 words to avoid tiny amendments and very long outliers. With these samples it will keep costs predictable and will still cover about 3/4 of the dataset. 


## SAMPLE INFO

**The goal:** 25 contracts, each between 1,000 and 15,000 words, where every one of the six risks has enough “Yes” examples to test fairly. Pure random picking could, by bad luck, give only one or two Non-Compete contracts, so we’ll guarantee a few for each risk first, then fill the rest randomly.

In [16]:
RISKS = [
    "Uncapped Liability",
    "Termination For Convenience",
    "Exclusivity",
    "Minimum Commitment",
    "Ip Ownership Assignment",
    "Non-Compete",
]

pool = df[(df["words"] >= 1000) & (df["words"] <= 15000)].copy()
print("Contracts in the 1,000–15,000 word range:", len(pool))

chosen = []
for risk in RISKS:
    yes = pool[(pool[f"{risk}-Answer"] == "Yes") & (~pool.index.isin(chosen))]
    chosen += yes.sample(3, random_state=42).index.tolist()

rest = pool[~pool.index.isin(chosen)]
chosen += rest.sample(25 - len(chosen), random_state=42).index.tolist()

sample = pool.loc[chosen]
print("Sample size:", len(sample))

sample[[f"{r}-Answer" for r in RISKS]].apply(lambda col: (col == "Yes").sum())

Contracts in the 1,000–15,000 word range: 375
Sample size: 25


Uncapped Liability-Answer              8
Termination For Convenience-Answer     9
Exclusivity-Answer                    15
Minimum Commitment-Answer             11
Ip Ownership Assignment-Answer        13
Non-Compete-Answer                    11
dtype: int64

In [17]:
cols = ["Filename", "words"] + [c for r in RISKS for c in (r, f"{r}-Answer")]
out = sample[cols].copy()
out.insert(1, "txt_path", sample["txt_path"].apply(lambda p: str(p.relative_to(".."))))

out.to_csv("../data/sample_25.csv", index=False)
print("Saved", len(out), "contracts |", out["words"].sum(), "words in total")

no_risk = (sample[[f"{r}-Answer" for r in RISKS]] != "Yes").all(axis=1).sum()
print("Contracts with none of the six risks:", no_risk)


Saved 25 contracts | 155905 words in total
Contracts with none of the six risks: 3


## SUMMARY

25 contracts, 155,905 words total (~200k tokens per full pass).
Yes-counts per risk: 8–15. 3 contracts contain none of the six risks.
